# Removing Duplicates

In **Notebook 05**, we removed exact duplicates and identified **natural key duplicates** — rows that share the same composite key (`school_urn + term + year` or `pupil_id + term + year`) but differ in other columns.

These are the harder duplicates to resolve because they require a **business rule** to decide which row to keep. This notebook:

1. **Inspects** the conflicting rows side by side to understand what differs
2. **Defines** a business rule for selecting the "winning" row
3. **Applies** `ROW_NUMBER` with meaningful ordering to keep exactly one row per key
4. **Audits** every discarded row in a separate table for traceability

### Source and output tables

| | Source | Deduplicated | Audit |
| --- | --- | --- | --- |
| **Schools** | `silver.schools_entire_duplicates_removed` | `silver.schools_deduplicated` | `silver.audit_schools_duplicate_rows` |
| **Pupils** | `silver.pupils_entire_duplicates_removed` | `silver.pupils_deduplicated` | `silver.audit_pupils_duplicate_rows` |

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

> **Prerequisite:** Run **Notebook 05 — Understanding Your Data** first to create the exact-deduplicated silver tables.

## Schools — detecting and resolving natural key conflicts

Exact duplicates are gone. The remaining conflicts are rows where the same school appears **twice in the same snapshot** with different values. Let's see which schools are affected and what actually differs between their rows.

First we'll create two fields `school_natural_key` & `school_occurance` to make observing both instances of the duplicate records easy.

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

In [0]:
-- Show conflicting school rows side by side
-- These share the same school_urn + term + year but differ in other columns
CREATE OR REPLACE TEMPORARY VIEW schools_keyed AS
SELECT row_id
      ,CONCAT_WS('|', school_urn, term, year) as school_natural_key
      ,ROW_NUMBER() OVER (PARTITION BY school_urn, term, year ORDER BY school_name) as school_occurance
      ,* EXCEPT (row_id)
FROM silver.schools_entire_duplicates_removed;

SELECT * 
FROM schools_keyed 
WHERE school_natural_key IN (SELECT school_natural_key FROM schools_keyed WHERE school_occurance > 1);

### Choosing a business rule — "most recent inspection within the snapshot year"

School **100005** (Willow Park) appears twice in autumn 2024 with two differences:

* **Name variant** — "Willow Park Academy" vs "Willow Park School"
* **Ofsted metadata** — one row has `Outstanding` with `last_inspection: 2024-03-01`, the other has `Good` with `last_inspection: 2023-03-22`

The conflict is fundamentally about **which Ofsted record is current**. The natural business rule is:

> **Keep the row whose `last_inspection` date falls within the snapshot year, preferring the most recent inspection.**

This is a strong rationale because:

* The `2024-03-01` inspection belongs to the same year as the autumn 2024 snapshot — it's the **current** record
* The `2023-03-22` inspection is from the **previous year** — likely stale data that wasn't updated
* It's grounded in domain logic rather than a generic proxy like string length
* The discarded row is preserved in an audit table, so the decision is reversible

We implement this with `ROW_NUMBER()`, ordering first by whether the inspection year matches the snapshot year, then by the inspection date itself (most recent first). 

We can remove and disregard the `school_occurance` field here as that was ordered arbitrarily on a field just so that we could observe the differences.

In [0]:
-- Rank schools within each natural key group
-- Prefer the row whose last_inspection falls within the snapshot year, then most recent
CREATE OR REPLACE TEMP VIEW schools_ranked AS
SELECT
  * EXCEPT (school_occurance)
  ,ROW_NUMBER() OVER (
    PARTITION BY school_urn, term, year
    ORDER BY
      CASE
        WHEN YEAR(TO_DATE(metadata_json:last_inspection)) = year THEN 0
        ELSE 1
      END
      ,TO_DATE(metadata_json:last_inspection) DESC
  ) AS dupe_rank
FROM schools_keyed;

SELECT school_natural_key, dupe_rank, school_urn, school_name, metadata_json
FROM schools_ranked 
WHERE school_natural_key IN (SELECT school_natural_key FROM schools_ranked WHERE dupe_rank > 1);

The `dupe_rank` column now marks each row: **rank 1** is the winning row we keep, **rank 2+** are the discarded duplicates. The next two cells split these into separate tables — the audit table captures every discarded row for traceability, and the deduplicated table retains only the winners.

We can now separate out duplicate and primary records into different datasets. The duplicate records are retained for audit purposes.

> **Note:** The dataset below is stored in `catalog_40_copper_analyst_training.silver.audit_schools_duplicate_rows`. As that catalog is read-only this notebook creates and displays it as a view.

In [0]:
-- Store the discarded rows (rank > 1) in an audit table for traceability
CREATE OR REPLACE TEMPORARY VIEW audit_schools_duplicate_rows AS
SELECT * EXCEPT(dupe_rank)
FROM schools_ranked
WHERE dupe_rank > 1;

SELECT * FROM audit_schools_duplicate_rows;

The rows where `dupe_rank = 1` become out main dataset with the correct grain. Note that we created a `school_natural_key` field above which concatenates `school_urn`, `term`, `year`. This show now be unique for every row.

We can drop the `dupe_rank` column from the final dataset as it no longer has any purpose.

> **Note:** The dataset below is stored in `catalog_40_copper_analyst_training.silver.schools_deduplicated`. As that catalog is read-only this notebook creates and displays it as a view.

In [0]:
-- Keep only the winning row (rank = 1) for each natural key
CREATE OR REPLACE TEMPORARY VIEW schools_deduplicated AS
SELECT * EXCEPT(dupe_rank)
FROM schools_ranked
WHERE dupe_rank = 1;

SELECT * FROM schools_deduplicated LIMIT 12;

In [0]:
-- Verify: every school_urn + term + year should now appear exactly once
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT school_natural_key) AS school_natural_keys
  ,COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', school_urn, term, year)) AS remaining_duplicates
FROM schools_deduplicated;

With `total_rows` matching `school_natural_keys` and zero remaining duplicates, the schools dataset now has exactly **one row per school per term per year** — the correct grain.

## Pupils — detecting and resolving natural key conflicts

The pupils data has a richer set of conflicts — 17 within-snapshot duplicates affecting 15 distinct pupils across all four snapshots. As we saw in Notebook 05, these include name variants, flipped FSM eligibility, changed SEN statuses, different parent contacts, and extra metadata fields.

Unlike schools — where the conflict was about which Ofsted inspection was current — the pupil conflicts are about **which record carries the most complete safeguarding and pastoral picture**. Let's inspect them.

First we'll create `pupil_natural_key` and `pupil_occurance` fields to simplify querying then observe the duplicated records.

In [0]:
CREATE OR REPLACE TEMPORARY VIEW pupils_keyed AS
SELECT row_id
      ,CONCAT_WS('|', pupil_id, term, year) as pupil_natural_key
      ,ROW_NUMBER() OVER(PARTITION BY pupil_id, term, year ORDER BY first_name) as pupil_occurance
      ,* EXCEPT (row_id)
FROM silver.pupils_entire_duplicates_removed;

SELECT * 
FROM pupils_keyed 
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_keyed WHERE pupil_occurance > 1)
ORDER BY pupil_natural_key;

### Choosing a business rule — prioritised domain logic

Looking across the 17 conflict pairs, four categories of difference emerge. Each maps to a domain priority that a school data team would recognise:

#### Priority 1 — Contact completeness (safeguarding)

Some rows have a parent **email address** while their counterpart doesn't (P011, P017). In a safeguarding context, the school needs as many communication channels as possible — a record with both phone *and* email is strictly more useful than one with phone alone.

This resolves the P011 conflict naturally: the David Jones row (father, Maidstone) carries an email address while the Emma Jones row (mother, Canterbury) does not — so the more contactable record wins. Similarly, P017 has one row with Robert Lewis's email and another without.

In [0]:
SELECT pupil_natural_key
      ,pupil_id
      ,first_name
      ,last_name 
      ,metadata_json:contact.parent_name AS contact_name
      ,metadata_json:contact.email AS contact_email
      ,metadata_json:contact.phone AS contact_phone
FROM pupils_keyed 
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_keyed WHERE pupil_occurance > 1)
      AND pupil_id IN ('P011','P017')
ORDER BY pupil_natural_key, pupil_occurance;

#### Priority 2 — SEN status level (precautionary principle)

Several pairs disagree on `sen_status` (P008, P010, P015, P020, P022). In UK schools, SEN provision follows a formal hierarchy: **EHCP > SEN Support > None**. An EHCP is a legally binding document; SEN Support reflects a formally identified need.

The precautionary principle applies: **prefer the higher level of support**. If a pupil's SEN status was genuinely discontinued, that should be updated through the proper SEND process — not inferred from a duplicate row. Keeping the higher status ensures no pupil loses documented support.

In [0]:
SELECT pupil_natural_key
      ,pupil_id
      ,first_name
      ,last_name 
      ,metadata_json:sen_status AS sen_status
FROM pupils_keyed 
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_keyed WHERE pupil_occurance > 1)
      AND pupil_id IN ('P008', 'P010','P015','P020','P022')
ORDER BY pupil_natural_key, pupil_occurance;

#### Priority 3 — Welfare documentation (pastoral completeness)

One row in each pair often carries additional fields the other lacks — `medical_notes`, `allergy_info`, `learning_support_plan`, `behaviour_notes`, `secondary_contact`, and similar. These are safety-critical: a missing allergy record or absent secondary contact could have real consequences.

We count the number of these welfare-relevant fields present and prefer the richer record. This also resolves P002's summer 2025 conflict, where one row carries a `behaviour_log` the other lacks.

In [0]:
SELECT pupil_natural_key
      ,pupil_id
      ,first_name
      ,last_name
      ,CAST(metadata_json:medical_notes IS NOT NULL AS INT)
        + CAST(metadata_json:allergy_info IS NOT NULL AS INT)
        + CAST(metadata_json:behaviour_notes IS NOT NULL AS INT)
        + CAST(metadata_json:behaviour_log IS NOT NULL AS INT)
        + CAST(metadata_json:assessment_notes IS NOT NULL AS INT)
        + CAST(metadata_json:learning_support_plan IS NOT NULL AS INT)
        + CAST(metadata_json:learning_support IS NOT NULL AS INT)
        + CAST(metadata_json:dietary_requirements IS NOT NULL AS INT)
        + CAST(metadata_json:secondary_contact IS NOT NULL AS INT)
        + CAST(metadata_json:exclusions IS NOT NULL AS INT)
      AS welfare_field_count
FROM pupils_keyed
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_keyed WHERE pupil_occurance > 1)
      AND pupil_id IN ('P002', 'P003', 'P005', 'P006', 'P007', 'P009', 'P013', 'P014', 'P016')
ORDER BY pupil_natural_key, pupil_occurance;

#### Priority 4 — Name length (data completeness tiebreaker)

P002 appears twice in autumn 2024 with different first names: `Bob` and `Robert`. Both rows are identical on every other priority — same email, same SEN status, same welfare field count. The longer name `Robert` is likely the more formal, complete form (`Bob` being a common shortening).

As a tiebreaker, prefer the row with the **longer combined name** (first + last). This favours the more complete record. Note that casing differences (e.g. `katie JONES` vs `Katie Jones`) are **not** addressed here — those are standardisation issues handled in the next notebook.

In [0]:
SELECT pupil_natural_key
      ,pupil_id
      ,first_name
      ,last_name
      ,LENGTH(first_name) + LENGTH(last_name) AS name_length
FROM pupils_keyed
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_keyed WHERE pupil_occurance > 1)
      AND pupil_id IN ('P002')
      AND year = 2024
ORDER BY pupil_natural_key, pupil_occurance;

#### Priority 5 — Deterministic tiebreaker (`row_id`)

After applying priorities 1–4, it is possible that some conflict pairs remain perfectly tied. Without a final tiebreaker, `ROW_NUMBER` will assign ranks **arbitrarily** — meaning the "winning" row could change between re-runs, across different compute engines, or if the data is migrated to another system.

This makes the pipeline **non-deterministic**, which is never acceptable: the same input data should always produce the same output. To guarantee a stable, repeatable result we use the `row_id` surrogate key as a last resort. The value itself carries no domain significance — it simply ensures the same row is always selected.

No conflicts in the current data reach Priority 5 — every pair is resolved by one of the four domain-grounded rules above. But the tiebreaker exists as a safety net: if future data introduces a fully tied pair, the pipeline will still produce deterministic results.

---

| Priority | Rule | Rationale | Conflicts resolved |
| --- | --- | --- | --- |
| **1** | Prefer row with parent email present | More communication channels = better safeguarding | P011, P017 |
| **2** | Prefer higher SEN status (EHCP > SEN Support > None) | Precautionary — don't lose legally significant support | P008, P010, P015, P020, P022 |
| **3** | Prefer row with more welfare documentation fields | Medical, allergy, and pastoral data is safety-critical | P002 (summer), P003, P005, P006, P007, P009, P013, P014, P016 |
| **4** | Prefer longer combined name (first + last) | Longer names tend to be the more formal, complete form | P002 (autumn) |
| **5** | Use `row_id` surrogate key (lowest wins) | Deterministic tiebreaker — ensures the same row is always picked | None currently — safety net |

### Applying the business rules

With five priorities defined, the implementation follows two steps:

1. **Score** — create a view that adds the `welfare_field_count` to each pupil row. This is computed upfront because Priority 3 needs to compare a derived value (the count of non-null welfare fields), and embedding that calculation inside a `ROW_NUMBER` window would make the logic harder to read and audit.

2. **Rank** — apply `ROW_NUMBER() OVER (PARTITION BY pupil_id, term, year ORDER BY ...)` against the scored view, with the five priorities expressed as successive `ORDER BY` columns. The first priority that differs between two rows determines the winner; later priorities only matter if all earlier ones are tied.

This separation keeps each step inspectable — we can verify the welfare scores independently before they feed into the ranking.

In [0]:
-- Step 1: Add welfare documentation field count to each pupil row
CREATE OR REPLACE TEMPORARY VIEW pupils_welfare_scored AS
SELECT * 
      ,(CASE WHEN metadata_json:medical_notes IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:allergy_info IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:annual_review_date IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:learning_support_plan IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:learning_support IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:behaviour_notes IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:behaviour_log IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:assessment_notes IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:dietary_requirements IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:secondary_contact IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:sen_review_notes IS NOT NULL THEN 1 ELSE 0 END
        + CASE WHEN metadata_json:exclusions IS NOT NULL THEN 1 ELSE 0 END
      ) AS welfare_field_count
FROM pupils_keyed;

SELECT pupil_natural_key, row_id, pupil_id, first_name, last_name, welfare_field_count
FROM pupils_welfare_scored
LIMIT 10;

With `welfare_field_count` now available on each row, we can apply the full five-priority `ROW_NUMBER` ranking. Each conflict pair is partitioned by `pupil_id + term + year`, and the priorities are evaluated in order until a winner emerges.

In [0]:
-- Step 2: Rank pupils using the five-priority cascade
CREATE OR REPLACE TEMPORARY VIEW pupils_ranked AS
SELECT *
      ,ROW_NUMBER() OVER (
        PARTITION BY pupil_id, term, year
        ORDER BY
          -- P1: Contact completeness — prefer row with parent email
          CASE WHEN metadata_json:contact.email IS NOT NULL THEN 0 ELSE 1 END
          -- P2: SEN status level — higher support wins (precautionary)
          ,CASE metadata_json:sen_status
            WHEN 'EHCP' THEN 0
            WHEN 'SEN Support' THEN 1
            ELSE 2
          END
          -- P3: Welfare documentation — more fields = richer pastoral picture
          ,welfare_field_count DESC
          -- P4: Name length — prefer longer (more complete) name
          ,LENGTH(first_name) + LENGTH(last_name) DESC
          -- P5: Deterministic tiebreaker — ensures the same row is always selected
          ,row_id ASC
      ) AS dupe_rank
FROM pupils_welfare_scored;

-- Preview: show conflict pairs with priority factors visible
SELECT pupil_natural_key
      ,dupe_rank
      ,row_id
      ,pupil_id
      ,first_name
      ,last_name
      ,metadata_json:contact.email AS has_email
      ,metadata_json:sen_status AS sen_status
      ,welfare_field_count
FROM pupils_ranked
WHERE pupil_natural_key IN (SELECT pupil_natural_key FROM pupils_ranked WHERE dupe_rank > 1)
ORDER BY pupil_natural_key, dupe_rank;

The preview shows each conflict pair resolved by the priority cascade: rows with email beat those without, then higher SEN status wins, then richer welfare documentation, then longer names favour the more complete form. The `row_id` tiebreaker exists as a safety net but is not needed for any current conflict.

> **Note:** The datasets below are stored in `catalog_40_copper_analyst_training.silver.audit_pupils_duplicate_rows` and `catalog_40_copper_analyst_training.silver.pupils_deduplicated`. As that catalog is read-only this notebook creates and displays them as views.

In [0]:
-- Store the discarded pupil rows for traceability
CREATE OR REPLACE TEMPORARY VIEW audit_pupils_duplicate_rows AS
SELECT * EXCEPT(pupil_occurance, dupe_rank)
FROM pupils_ranked
WHERE dupe_rank > 1;

SELECT * FROM audit_pupils_duplicate_rows;

In [0]:
-- Keep only the winning row for each pupil + term + year
CREATE OR REPLACE TEMPORARY VIEW pupils_deduplicated AS
SELECT * EXCEPT(dupe_rank)
FROM pupils_ranked
WHERE dupe_rank = 1;

SELECT * FROM pupils_deduplicated LIMIT 12;

In [0]:
-- Verify: every pupil_id + term + year should now appear exactly once
SELECT
  COUNT(*) AS total_rows
  ,COUNT(DISTINCT pupil_natural_key) AS pupil_natural_keys
  ,COUNT(*) - COUNT(DISTINCT CONCAT_WS('|', pupil_id, term, year)) AS remaining_duplicates
FROM pupils_deduplicated;

## Summary

Natural key duplicates are rows that share the same identifying key but differ in other columns. Unlike exact duplicates, resolving them requires a **business decision** grounded in domain knowledge:

| Dataset | Business rule | Rationale |
| --- | --- | --- |
| **Schools** | Keep the row whose `last_inspection` falls within the snapshot year | The current Ofsted record is the authoritative one |
| **Pupils** | Prioritised cascade: contact completeness → SEN level → welfare documentation → name length → `row_id` | Each factor reflects a safeguarding or data quality priority |

The process for both:

1. **Inspect** — view conflicting rows side by side to understand what differs
2. **Choose** — define business rules grounded in domain knowledge, with clear priority ordering
3. **Apply** — `ROW_NUMBER() OVER (PARTITION BY key ORDER BY rules)` ranks each group
4. **Audit** — discarded rows stored in `audit_schools_duplicate_rows` and `audit_pupils_duplicate_rows`
5. **Persist** — winning rows saved to `schools_deduplicated` and `pupils_deduplicated`

Both datasets now have exactly **one row per entity per term per year** — the correct grain for downstream analysis.

### What's next

* **Notebook 07 — Standardising Fields and Labels** will clean the values themselves (casing, whitespace, abbreviations)
* **Notebook 08 — Building the Gold Layer** will apply all steps end-to-end and write to gold